# # 5 Germ layer specification dynamics

## Purpose

This notebook creates vizualization to explore the dynamics of germ layer specification and highlight how the lineage tracing data is analyzed.

In [1]:
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import seaborn as sns
from devmap.config import embryo_palette, germ_layer_palette, lineage_palette, set_theme, stage_palette
from devmap.fate import (
    annotate_commitment,
    commitment_index,
    extant_node_attribute,
    extant_transition_count,
    permutation_test_commitment,
)
from devmap.plots import plot_area, plot_restriction_sankey, plot_simplex_triangle, simplex_sort_key, simplex_to_hex
from devmap.progenitors import identify_fate_progenitors
from devmap.utils import get_paths, load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("fate")

## Load data

In [2]:
tdata = load_data("topology")
embryos = pd.read_csv(base_path / "validation/results/lineage_stats.csv", index_col=0)

## Identify fate-restricted clades

In [3]:
one_hot = pd.get_dummies(tdata.obs["germ_layer"]).astype(int)
tdata.obsm["germ_layer_counts"] = one_hot
tdata.obs["n"] = 1
py.tl.ancestral_states(tdata, keys = "germ_layer_counts", method = "sum")
py.tl.ancestral_states(tdata, keys = "n", method = "sum")

In [4]:
clone_progenitors = {}
leaf_assignments = []
for clone, tree in tdata.obst.items():
    print(f"Processing clone {clone}...")
    progenitors, clone_assignements= identify_fate_progenitors(
        tree,
        "germ_layer_counts",
        fate_names = one_hot.columns)
    clone_progenitors[clone] = progenitors.assign(clone=clone)
    leaf_assignments.append(clone_assignements)
progenitors = pd.concat(clone_progenitors.values())
leaf_assignments = pd.concat(leaf_assignments)
progenitors["stage"] = progenitors["clone"].str.split("-R").str[0]
progenitors["embryo"] = progenitors["clone"].str.split("-C").str[0]

Processing clone E7.5-R1-C1...
Processing clone E7.5-R1-C2...
Processing clone E7.5-R2-C1...
Processing clone E7.5-R2-C2...
Processing clone E7.5-R2-C3...
Processing clone E7.5-R2-C4...
Processing clone E7.5-R2-C5...
Processing clone E7.5-R2-C6...
Processing clone E7.5-R3-C1...
Processing clone E7.5-R3-C2...
Processing clone E7.5-R3-C3...
Processing clone E8.0-R1-C1...
Processing clone E8.0-R1-C2...
Processing clone E8.0-R1-C3...
Processing clone E8.0-R2-C1...
Processing clone E8.0-R2-C2...
Processing clone E8.0-R3-C1...
Processing clone E8.0-R3-C2...
Processing clone E8.0-R3-C3...
Processing clone E8.0-R3-C4...
Processing clone E8.0-R3-C5...
Processing clone E8.0-R3-C6...
Processing clone E8.5-R1-C1...
Processing clone E8.5-R1-C2...
Processing clone E8.5-R1-C3...
Processing clone E8.5-R1-C4...
Processing clone E8.5-R2-C1...
Processing clone E8.5-R2-C2...
Processing clone E8.5-R3-C1...
Processing clone E8.5-R3-C2...
Processing clone E8.5-R3-C3...
Processing clone E9.0-R1-C1...
Processi

## Clade stats

In [5]:
prog_stats = (
    progenitors
    .query("total_descendants > 1 & stage != 'E10.0'")
    .sort_values("time")
    .groupby(["embryo", "stage", "fate"])
    .apply(lambda g: pd.Series({
        "mean_time": g["time"].mean(),
        "mean_size": g["fate_descendants"].mean(),
        "n_progenitors": g["node"].nunique()
    }))
    .reset_index()
)
prog_stats = prog_stats.query("fate.isin(['Ectoderm', 'Mesoderm', 'Endoderm'])")
prog_stats["fate"] = pd.Categorical(prog_stats ["fate"], categories=["Endoderm", "Ectoderm", "Mesoderm"], ordered=True)

/tmp/ipykernel_3598878/1708985036.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [6]:
progenitors.to_csv(results_path / "germ_layer_progenitors.csv")
leaf_assignments.to_csv(results_path / "germ_layer_progenitor_descendants.csv")
prog_stats.to_csv(results_path / "germ_layer_progenitor_stats.csv")

## Plot clade stats

Number of clades

In [11]:
fig, axes = plt.subplots(3,1,figsize=(.8, 3), dpi=600, sharex=True)
for i, fate in enumerate(["Ectoderm", "Mesoderm", "Endoderm"]):
    df = prog_stats.query("fate == @fate")
    sns.barplot(data = df, y = "stage", x = "mean_size", hue = "fate",legend = False,
                    palette=germ_layer_palette, saturation = 1, ax=axes[i],edgecolor="black",
                    linewidth=0.5,errorbar="se",capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"}) 
    sns.scatterplot(data = df, y = "stage", x = "mean_size", color = "black", s = 6, 
                    legend = False, ax=axes[i], edgecolor="none")
    axes[i].set_xlabel("Mean population size")
    axes[i].set_ylabel(fate)
    axes[i].set_yticks([])
    axes[i].set_xticks([0, 50, 100])
save_plot(plots_path / "germ_layer_prog_size.svg", fig, rasterize=False)

Clade size

In [12]:
fig, axes = plt.subplots(3,1,figsize=(.8, 3), dpi=600, sharex=True)
for i, fate in enumerate(["Ectoderm", "Mesoderm", "Endoderm"]):
    df = prog_stats.query("fate == @fate")
    sns.barplot(data = df, y = "stage", x = "n_progenitors", hue = "fate",legend = False,
                    palette=germ_layer_palette, saturation = 1, ax=axes[i],edgecolor="black",
                    linewidth=0.5,errorbar="se",capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"}) 
    sns.scatterplot(data = df, y = "stage", x = "n_progenitors", color = "black", s = 6, 
                    legend = False, ax=axes[i], edgecolor="none")
    axes[i].set_xlabel("Number of progenitors")
    axes[i].set_ylabel(fate)
    axes[i].set_yticks([])
    axes[i].set_xticks([0, 1000, 2000])
save_plot(plots_path / "germ_layer_n_prog.svg", fig, rasterize=False)

Clade size distribution

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(6.2, 1.35), dpi=600, sharex=True, sharey=False)
df["stage"] = pd.Categorical(df["stage"], categories=list(reversed(["E7.5", "E8.0", "E8.5", "E9.0", "E9.5"])), ordered=True)
for i, fate in enumerate(["Endoderm", "Mesoderm", "Ectoderm"]):
    sns.histplot(data = progenitors.query("fate == @fate & stage != 'E10.0' & fate_descendants > 1"),bins = 15, alpha = 1,ax = axes[i],linewidth = 0,
                x = "fate_descendants", hue = "stage", palette=stage_palette, legend = False, log_scale=True, multiple="dodge")
    axes[i].set_ylabel("")
    axes[i].set_xlabel("Progenitor output")
    axes[i].set_title(fate)
save_plot(plots_path / "germ_layer_prog_output_dist.svg", fig, rasterize=False) 

Time

In [22]:
fig, axes = plt.subplots(3,1,figsize=(.9, 3), dpi=600, sharex=True)
for i, fate in enumerate(["Ectoderm", "Mesoderm", "Endoderm"]):
    df = prog_stats.query("fate == @fate")
    sns.boxplot(data = df, y = "stage", x = "mean_time", hue = "fate",legend = False,
                    palette=germ_layer_palette, saturation = 1, ax=axes[i],
                    linewidth=0.5, linecolor="black") 
    sns.scatterplot(data = df, y = "stage", x = "mean_time", color = "black", s = 6, 
                    legend = False, ax=axes[i], edgecolor="none", zorder = 10)
    axes[i].set_xlabel("Mean time")
    axes[i].set_ylabel(fate)
    axes[i].set_yticks([])
save_plot(plots_path / "germ_layer_prog_time.svg", fig, rasterize=False)

Time distribution

In [ ]:
fig, axes = plt.subplots(3,1,figsize=(.9, 3), dpi=600, sharex=True)
for i, fate in enumerate(["Endoderm", "Mesoderm", "Ectoderm"]):
    sns.kdeplot(data = progenitors.query("fate == @fate & stage != 'E10.0' & fate_descendants > 1"),
             x = "time", hue = "stage", palette=stage_palette, ax=axes[i], legend = False)
    axes[i].set_xlabel("Time")
    axes[i].set_ylabel("")
    axes[i].set_xticks([4,6,8,10])
    axes[i].set_xlim(4, 10)
    axes[i].set_ylim(0, .2)
    axes[i].axvspan(6, 7.5, color="gray", alpha=0.2, edgecolor="none",linewidth=0)
save_plot(plots_path / "germ_layer_prog_time_dist.svg", fig, rasterize=False)

clade percent

In [37]:
plot_df = (
    prog_stats
    .pivot(index="embryo", columns="fate", values="frac")
    .fillna(0)
)

fate_order = [f for f in germ_layer_palette if f in plot_df.columns]
plot_df = plot_df[fate_order]

fig, ax = plt.subplots(figsize=(2.4, 1.8), dpi=600)

bottom = np.zeros(len(plot_df))

for fate in fate_order:
    ax.bar(
        plot_df.index,
        plot_df[fate],
        bottom=bottom,
        label=fate,
        color=germ_layer_palette[fate],
        edgecolor="black",
        linewidth=0.5,
    )
    bottom += plot_df[fate].to_numpy()

ax.set_xlabel("")
ax.set_ylabel("Clade distribution (%)")
ax.set_ylim(0, 100)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
save_plot(plots_path / "germ_layer_clade_distribution.svg", fig, rasterize=False)

## Number of clades vs chimerism

In [5]:
stats = progenitors.query("fate != 'Epiblast' & fate_descendants > 1").embryo.value_counts().reset_index(name="n_progenitors").set_index("embryo")
stats = stats.merge(embryos, left_index=True, right_index=True).query("embryo != 'E10.0-R1'")
stats["stage"] = stats.index.str.split("-").str[0]

In [135]:
stats["log_cell_total_estimate"] = np.log10(stats["cell_total_estimate"])
stats["replicate"] = stats.index.str.split("-").str[1]

fig, ax = plt.subplots(figsize=(1.8, 1.8))

sns.regplot(
    data=stats,
    x="log_cell_total_estimate",
    y="n_progenitors",
    scatter=False,
    ax=ax,
    color="black",
    line_kws={"linewidth": 1, "zorder": 1}
)

sns.scatterplot(
    data=stats,
    x="log_cell_total_estimate",
    y="n_progenitors",
    hue="chimerism_rate",
    palette="viridis",
    hue_norm=mcolors.Normalize(vmin=0.5, vmax=1),
    ax=ax,
)

# add replicate labels
for i, row in stats.iterrows():
    ax.text(
        row["log_cell_total_estimate"],
        row["n_progenitors"],
        row["replicate"],
        fontsize=6,
        ha="center",
        va="bottom",
    )

ax.set_xlabel("Total cell estimate")
ax.set_ylabel("Total germ layer-restricted clades")

ax.set_xticks(np.log10([1e3, 1e4, 1e5, 1e6]))
ax.set_xticklabels(["$10^3$", "$10^4$", "$10^5$", "$10^6$"])

plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

save_plot(plots_path / "germ_layer_clade_count_vs_chimerism.svg", fig)

## Sankey diagram

In [6]:
stage_tdata = tdata[(tdata.obs.stage == "E9.5") & tdata.obs.clone.notnull()].copy()

In [57]:
germ_assignment = stage_tdata.obs["germ_layer"].replace({"Epiblast": "Other","Primordial germ cell": pd.NA}).to_dict()
for tree in stage_tdata.obst.values():
    annotate_commitment(tree, [germ_assignment], threshold=0.9, exclude_categories=["Other"])
paths = extant_transition_count(stage_tdata, key = "commitment", depth_key="time", bins = np.arange(0, 10, .5))
paths["count"] = np.ceil(paths["count"] / 3).astype(int)

In [134]:
fig, ax = plt.subplots(figsize=(3.2, 2),dpi=600)
sankey_order = ["Uncommitted", "Ectoderm","Mesoderm", "Endoderm"]
sankey_palette = sankey_palette = {**germ_layer_palette,**{"Uncommitted": "#CCCCCC"}}
plot_restriction_sankey(paths, normalize = True, order = sankey_order, palette = sankey_palette, ax = ax)
plt.xlim(0, 20)
plt.xticks([0, 4, 8, 12, 16, 20], ["0", "2", "4", "6", "8", "10"]);
plt.xticks(rotation=0)
save_plot(plots_path / "germ_layer_sankey.svg", fig)

## Ancestral linkage

In [398]:
for target in ["Mesoderm", "Endoderm", "Ectoderm", "Epiblast"]:
    py.tl.ancestral_linkage(tdata, groupby="germ_layer", target = target, depth_key="time",
                permutation_mode="non_target", by_tree = True, n_permutations=1)
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
for target in ["Mesoderm", "Endoderm", "Ectoderm", "Epiblast"]:
    norm = mcolors.TwoSlopeNorm(vmin=-2, vcenter=0, vmax=3)
    sc.pl.umap(tdata[tdata.obs[f"{target}_linkage"].notnull()], color=f"{target}_linkage", s = .2, cmap="RdBu_r", norm=norm, sort_order=False,
        colorbar_loc = None, frameon=False, title = "", ax = ax, show = False)
    save_plot(plots_path / f"{target}_linkage_umap.svg", fig, rasterize = True)

## Progenitor stats by cell type

In [46]:
tdata.obs["progenitor"] = leaf_assignments["progenitor"]
tdata.obs["progenitor_fate"] = leaf_assignments["fate"]
tdata.obs["clone_depth"] = tdata.obs["progenitor"].map(progenitors.groupby("node")["time"].first())
prog_by_type = tdata.obs.query("stage != 'E10.0'").groupby(["germ_layer","stage","cell_type"], observed=True)[
    ["Epiblast_linkage","clone_depth"]].mean().reset_index()

In [576]:
fig, ax = plt.subplots(figsize=(2, 2), dpi=600, layout="constrained")
palette = germ_layer_palette.copy()
for germ_layer in ["Ectoderm", "Mesoderm", "Endoderm"]:
    df = prog_by_type.query("stage == 'E9.5' & germ_layer == @germ_layer")
    if germ_layer == "Endoderm":
        palette = {"Endoderm":"black"}
    else:
        palette = germ_layer_palette
    sns.regplot(data = df, color = palette[germ_layer],line_kws={"linewidth": "1"},
        y = "Epiblast_linkage", x = "clone_depth", scatter = False, ax=ax)
    sns.scatterplot(data = df, color = germ_layer_palette[germ_layer], edgecolor="black",
                    y = "Epiblast_linkage", x = "clone_depth", s = 10, legend = False, ax=ax, zorder = 2)
plt.ylabel("Mean epiblast linkage")
plt.xlabel("Mean germ layer commitment (day)")
save_plot(plots_path / "linkage_vs_depth_by_cell_type.svg", fig)

## Progenitor increase by cell type

In [67]:
prog_counts_with_type = tdata.obs.groupby(["progenitor","stage"]).agg(
    n = ("n", "sum"),
    germ_layer = ("germ_layer", lambda x: x.mode()[0]),
    cell_type = ("cell_type", lambda x: x.mode()[0])).reset_index()
prog_counts_with_type = prog_counts_with_type.query("stage != 'E10.0' & n > 1").groupby(
    ["germ_layer","stage","cell_type"], observed=True).size().reset_index(name="n_progenitors")
prog_by_type_pivot = prog_counts_with_type.pivot_table(
    index=["germ_layer", "cell_type"], columns="stage", values="n_progenitors").fillna(0).reset_index()

In [93]:
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
for germ_layer in ["Ectoderm", "Mesoderm", "Endoderm"]:
    sns.scatterplot(data = prog_by_type_pivot.query("germ_layer == @germ_layer"), x = "E9.0", y = "E9.5", ax = ax,
                    color = germ_layer_palette[germ_layer], legend = False, edgecolor="black", s = 15)
# add dashed line y=x
plt.plot([0, 1000], [0, 1000], linestyle="--", color="black", zorder = -1)
save_plot(plots_path / "e9_vs_e95_prog_counts.svg", fig, rasterize=False)

## Germ layer fate bias

In [7]:
germ_layer_fate = extant_node_attribute(tdata, "germ_layer_counts", bins = np.arange(0, 10.5, 0.5), depth_key="time", column_names=one_hot.columns)
germ_layer_fate.obs["color"] = simplex_to_hex(germ_layer_fate[:,["Ectoderm", "Mesoderm", "Endoderm"]].X)
germ_layer_fate.obs["n"] = germ_layer_fate.X.sum(axis=1)
commitment_index(germ_layer_fate, key_added="commitment_index")
germ_layer_fate.obs["embryo"] = germ_layer_fate.obs["tree"].str.split("-C").str[0]
germ_layer_fate.obs["stage"] = germ_layer_fate.obs["embryo"].str.split("-R").str[0]

In [290]:
germ_layer_fate.obs["n"] = germ_layer_fate.X.sum(axis=1)
g = germ_layer_fate.obs
group_cols = ["time", "embryo","stage"] 
df = (
    (g["commitment_index"] * g["n"])
    .groupby([g[c] for c in group_cols])
    .sum()
    / g["n"].groupby([g[c] for c in group_cols]).sum()
).reset_index(name="commitment_index")

In [292]:
fig, axes = plt.subplots(2,1,figsize=(2,3), dpi=600, sharex=True)
sns.lineplot(data=df.query("stage != 'E10.0'"), x="time", y="commitment_index",
            hue = "stage", palette=stage_palette, ax = axes[1])
axes[1].axvspan(6, 7.5, color="gray", alpha=0.2, edgecolor="none",linewidth=0)
axes[1].set_ylabel("Germ layer\ncommitment index");
axes[1].set_xlabel("Developmental time (days)");
plot_area(germ_layer_fate.obs.query("stage == 'E8.5'"), ax=axes[0], sort_fn = simplex_sort_key, percent=True)
axes[0].set_ylabel("Germ layer\nFate distribution");
axes[0].set_xticks([0, 2, 4, 6, 8, 10]);
axes[0].set_xlim(0, 10)
save_plot(plots_path / "germ_layer_commitment.svg", fig, rasterize=True)

## Permutation test for germ layer fate bias

In [22]:
e95_fate = germ_layer_fate[germ_layer_fate.obs["stage"] == "E9.5"].copy()
stats, permutations = permutation_test_commitment(e95_fate, n_permutations=100)

In [80]:
fig, ax = plt.subplots(figsize=(2, 1.35), dpi=600)
sns.lineplot(data=stats, x="time", y="commitment_index", ax = ax, hue = "embryo", palette=embryo_palette)
sns.lineplot(data=permutations, x="time", y="commitment_index", color="black", alpha=1, ax = ax, legend=False)
plt.ylabel("Germ layer\ncommitment index")
ax.axvspan(6, 7.5, color="gray", alpha=0.2, edgecolor="none",linewidth=0)
# add dashed vertical line at time = 4
ax.axvline(x=4, color="black", linestyle="--", linewidth=1)
ax.set_xticks([0, 2, 4, 6, 8, 10]);
# legend outside of plot
ax.legend(title="Embryo", bbox_to_anchor=(1.05, 1), loc='upper left')
save_plot(plots_path / "e95_commitment_permutation_test.svg", fig, rasterize=False)

## Example tree

In [8]:
clone_tdata = tdata[tdata.obs["clone"] == "E8.5-R1-C1"].copy()
# Color edges
for edge in clone_tdata.obst["E8.5-R1-C1"].edges:
    parent, child = edge
    germ_layer_counts = np.array(clone_tdata.obst["E8.5-R1-C1"].nodes[child].get("germ_layer_counts", None))
    frac = germ_layer_counts[[3,0,1]]
    if frac.sum() > 0:
        frac = frac / frac.sum()
        color = str(simplex_to_hex(frac))
    else:
        color = "#D3D3D3"
    clone_tdata.obst["E8.5-R1-C1"].edges[parent, child]["color"] = color
# Expand epliblast so it is visible in the plot
mask = clone_tdata.obs["germ_layer"].eq("Epiblast")
expanded = (
    mask
    | mask.shift(1, fill_value=False)
    | mask.shift(2, fill_value=False)
    | mask.shift(-1, fill_value=False)
    | mask.shift(-2, fill_value=False)
)
clone_tdata.obs["is_epiblast"] = expanded
# Annotate fate-restricted clades
progenitor_nodes = set(progenitors.node)
for node in clone_tdata.obst["E8.5-R1-C1"].nodes:
    if node not in progenitor_nodes:
        continue
    fate = progenitors.query("node == @node").fate.values[0]
    if node in progenitor_nodes and fate in ["Ectoderm", "Mesoderm", "Endoderm"]:
        clone_tdata.obst["E8.5-R1-C1"].nodes[node]["progenitor"] = fate
    else:
        if "progenitor" in clone_tdata.obst["E8.5-R1-C1"].nodes[node]:
            del clone_tdata.obst["E8.5-R1-C1"].nodes[node]["progenitor"]

Tree with germ layer annotations

In [17]:
fig, ax = plt.subplots(figsize=(3, 3.6), dpi = 600)
py.pl.tree(clone_tdata, depth_key="time", tree = "E8.5-R1-C1", ax = ax, branch_linewidth=.8)
py.pl.annotation(clone_tdata, keys = ["germ_layer"], palette = germ_layer_palette, label = "")
save_plot(plots_path / "e85_germ_layer_tree_annotates.svg", fig, rasterize=True)

Tree with germ layer annotation and edge coloring

In [18]:
fig, ax = plt.subplots(figsize=(3, 3.6), dpi = 600)
py.pl.tree(clone_tdata, depth_key="time", branch_color="color", tree = "E8.5-R1-C1", ax = ax, branch_linewidth=.8)
py.pl.annotation(clone_tdata, keys = ["germ_layer"], palette = germ_layer_palette, label = "")
save_plot(plots_path / "e85_germ_layer_tree_no_nodes.svg", fig, rasterize=True)

Tree with germ layer annotations, edge coloring, and nodes highlighted

In [134]:
fig, ax = plt.subplots(figsize=(3, 3.6), dpi = 600)
py.pl.tree(clone_tdata, depth_key="time", branch_color="color", tree = "E8.5-R1-C1", ax = ax, branch_linewidth=.8)
py.pl.annotation(clone_tdata, keys="is_epiblast", label = False, legend = False, palette = {True: germ_layer_palette["Epiblast"], False: "lightgrey"})
#py.pl.annotation(clone_tdata, keys="Epiblast_linkage", label = False, legend = False, gap = .01,vmax = 3, vmin = -2, cmap="RdBu_r")
py.pl.nodes(clone_tdata, color="progenitor", palette=germ_layer_palette, size=7, legend = False, outline_width=.3)
save_plot(plots_path / "e85_germ_layer_tree.svg", fig, rasterize=True)

Example clade

In [146]:
fig, ax = plt.subplots(figsize=(4, 1), dpi = 1600)
i = 210
py.pl.tree(clone_tdata[i:i+87], depth_key="time", branch_color="color", tree = "E8.5-R1-C1", ax = ax, branch_linewidth=.6)
py.pl.nodes(clone_tdata, color="progenitor", palette=germ_layer_palette, size=7, legend = False, outline_width=.3)
save_plot(plots_path / "e85_germ_layer_tree_zoom.svg", fig, rasterize=True)

Colormap

In [299]:
fig, ax = plt.subplots(figsize=(1, 1))
plot_simplex_triangle(ax = ax)
save_plot(plots_path / "simplex_triangle.svg", fig, rasterize=True)